# Módulo 2: Indexación, Slicing y Filtrado Booleano

## 🎯 Objetivos de Aprendizaje
Al finalizar este cuaderno, serás capaz de:
1. Extraer elementos individuales, filas, columnas y submatrices mediante indexación posicional y *slicing* multidimensional (`[filas, columnas]`).
2. Distinguir cuándo NumPy devuelve una **vista (*View*)** compartida en memoria frente a una **copia independiente (*Copy*)**, evitando corrupciones accidentales en los datos originales.
3. Filtrar arreglos de forma vectorizada mediante **máscaras booleanas** simples y compuestas (`&`, `|`, `~`), así como aplicar condicionales matriciales con `np.where()`.
4. Utilizar *Fancy Indexing* para extraer subconjuntos no contiguos en un orden arbitrario.

---

## 🗺️ Mapa Mental del Módulo

```text
                   [ Acceso y Selección en ndarray ]
                                   │
       ┌───────────────────────────┼───────────────────────────┐
       ▼                           ▼                           ▼
1. POSICIONAL Y SLICING     2. MEMORIA (COPY vs VIEW)   3. INDEXACIÓN AVANZADA
├── 1D: vec[inicio:fin:paso]├── Slicing -> VIEW (comparte)├── Máscaras Booleanas (&, |, ~)
├── 2D: mat[filas, cols]    ├── .copy() -> COPY (aísla)   ├── np.where(cond, A, B)
└── Índices negativos (-1)  └── np.shares_memory(A, B)    └── Fancy Indexing: mat[[0, 2]]

In [1]:
## Importar numpy
import numpy as np

## 2.1 Indexación Posicional y Slicing (`[filas, columnas]`)

En NumPy, el acceso a matrices 2D se realiza separando las dimensiones mediante una coma dentro de un único par de corchetes: `matriz[fila, columna]`.

### Sintaxis de Slicing
La regla general para rebanar cualquier eje es `inicio:fin:paso`:
* `inicio`: Índice donde comienza el corte (**incluido**). Si se omite, asume `0`.
* `fin`: Índice donde termina el corte (**excluido**). Si se omite, llega hasta el final.
* `paso`: Incremento entre elementos. Si es negativo (`-1`), invierte el orden.
* Dos puntos solos (`:`): Selecciona **todos** los elementos de ese eje.

### Sintaxis Mínima

In [5]:
# Vector 1D: Consumo energético diario (kWh) durante 12 días
consumo_diario = np.array([14.2, 15.0, 13.8, 16.5, 18.1, 17.9, 12.4, 13.1, 15.6, 16.2, 19.0, 14.8])

print("\nVector completo:", consumo_diario)
print("Primeros 4 días:", consumo_diario[:4])
print("Últimos 3 días (índices negativos):", consumo_diario[-3:])
print("Días alternos (paso 2):", consumo_diario[::2])

# Matriz 2D (4x5): Registro de temperatura (°C) de 4 sensores (filas) en 5 horas (columnas)
sensores = np.array([
    [20.1, 21.3, 22.8, 24.0, 23.5],  # Sensor 0
    [19.8, 20.5, 21.9, 23.1, 22.7],  # Sensor 1
    [25.0, 26.2, 27.5, 28.9, 28.0],  # Sensor 2
    [18.2, 18.9, 19.5, 20.4, 20.0]   # Sensor 3
])

print("\nMatriz completa (4x5):\n", sensores)

# 1. Fila completa: Devuelve vector 1D de shape (5,)
print("\nFila 2 completa:", sensores[2, :])

# 2. Columna completa: Devuelve vector 1D de shape (4,)
print("Última columna:", sensores[:, -1])

# 3. Submatriz central(índices 1:3 y 1:4)
submatriz = sensores[1:3, 1:4]
print("\nSubmatriz extraída (shape", submatriz.shape, "):\n", submatriz)


Vector completo: [14.2 15.  13.8 16.5 18.1 17.9 12.4 13.1 15.6 16.2 19.  14.8]
Primeros 4 días: [14.2 15.  13.8 16.5]
Últimos 3 días (índices negativos): [16.2 19.  14.8]
Días alternos (paso 2): [14.2 13.8 18.1 12.4 15.6 19. ]

Matriz completa (4x5):
 [[20.1 21.3 22.8 24.  23.5]
 [19.8 20.5 21.9 23.1 22.7]
 [25.  26.2 27.5 28.9 28. ]
 [18.2 18.9 19.5 20.4 20. ]]

Fila 2 completa: [25.  26.2 27.5 28.9 28. ]
Última columna: [23.5 22.7 28.  20. ]

Submatriz extraída (shape (2, 3) ):
 [[20.5 21.9 23.1]
 [26.2 27.5 28.9]]


:::{note} Regla de Reducción Dimensional
Cuando indexas un eje con un **número entero escalar** (ej. `sensores[2, :]`), ese eje desaparece y el resultado baja de rango (de matriz 2D a vector 1D). Cuando indexas ambos ejes con **rangos o slices** (ej. `sensores[2:3, :]`), se preserva la estructura 2D (`shape = (1, 5)`).
:::

:::{warning} Trampa Común: Doble corchete `matriz[i][j]` vs. coma `matriz[i, j]`
Aunque `sensores[2][3]` funciona, **no debe usarse**. El doble corchete crea primero un arreglo temporal intermedio con toda la fila `2` y luego indexa el elemento `3`, siendo más lento y fallando al intentar extraer columnas (`sensores[:][3]` NO extrae la columna 3, sino la fila 3). Usa siempre `sensores[2, 3]`.
:::

### 🧩 Micro-reto 2.1
Dada una matriz `tablero` de $6 \times 6$ creada con `np.arange(1, 37).reshape(6, 6)`:
1. Extrae la **esquina inferior derecha** de tamaño $3 \times 3$.
2. Extrae los elementos de la **primera columna** en orden inverso (de abajo hacia arriba).

In [6]:
tablero = np.arange(1, 37).reshape(6, 6)
# Escribe tu solución aquí:

## 2.2 Gestión de Memoria: Copy vs. View

Por razones de eficiencia y velocidad, **el *slicing* básico en NumPy nunca duplica los datos en la memoria RAM**. En su lugar, devuelve una **vista (*View*)**: una ventana que apunta exactamente a las mismas direcciones de memoria que el arreglo original.

* **View (Vista):** Cualquier modificación hecha sobre la vista **altera directamente al arreglo original** (y viceversa). Ocurre con *slicing* básico (`arr[1:4]`) y `.reshape()`.
* **Copy (Copia):** Crea un bloque nuevo e independiente en memoria. Modificar la copia no afecta al original. Se fuerza explícitamente con `.copy()`.

### Sintaxis Mínima

In [12]:
# Matriz original de calificaciones (3x4)
notas_originales = np.array([
    [7.0, 8.0, 6.5, 9.0],
    [5.0, 6.0, 5.5, 7.0],
    [9.5, 9.0, 8.5, 10.0]
])

print("\nMatriz original de calificaciones:\n", notas_originales)

# 1. Creamos un slice (esto genera una VISTA)
vista_estudiante_1 = notas_originales[1, :]

# 2. Creamos una COPIA independiente
copia_estudiante_1 = notas_originales[1, :].copy()

# Verificamos si comparten memoria física
print("\n¿La vista comparte memoria con el original?:", np.shares_memory(notas_originales, vista_estudiante_1))
# Modificamos la VISTA
vista_estudiante_1[0] = 10.0
print("Notas originales tras modificar la VISTA (¡fila 1 alterada!):\n", notas_originales)

# Verificamos si comparten memoria física
print("\n¿La copia comparte memoria con el original?:", np.shares_memory(notas_originales, copia_estudiante_1))
# Modificamos la COPIA
copia_estudiante_1[1] = 0.0
print("Notas originales tras modificar la COPIA (intactas):\n", notas_originales)


Matriz original de calificaciones:
 [[ 7.   8.   6.5  9. ]
 [ 5.   6.   5.5  7. ]
 [ 9.5  9.   8.5 10. ]]

¿La vista comparte memoria con el original?: True
Notas originales tras modificar la VISTA (¡fila 1 alterada!):
 [[ 7.   8.   6.5  9. ]
 [10.   6.   5.5  7. ]
 [ 9.5  9.   8.5 10. ]]

¿La copia comparte memoria con el original?: False
Notas originales tras modificar la COPIA (intactas):
 [[ 7.   8.   6.5  9. ]
 [10.   6.   5.5  7. ]
 [ 9.5  9.   8.5 10. ]]


### 🧩 Micro-reto 2.2
1. Crea una matriz `original` de $4 \times 4$ llena de unos (`np.ones((4, 4))`).
2. Extrae el bloque central de $2 \times 2$ en una variable llamada `centro` **sin** usar `.copy()`, y asígnale el valor `9`. Imprime `original` para comprobar el cambio.
3. Ahora repite el proceso extrayendo la primera fila en `fila_segura` usando `.copy()`, cámbiala a `5`, y verifica con `np.shares_memory()` que son independientes.

In [13]:
# Escribe tu solución aquí:

## 2.3 Máscaras Booleanas, `np.where` y Fancy Indexing

Cuando no conocemos las posiciones exactas de los datos, filtramos por **condiciones lógicas**. Al evaluar una comparación sobre un `ndarray` (ej. `matriz > 25`), NumPy genera una **máscara booleana**: un arreglo del mismo `shape` lleno de `True` y `False`.

### Operadores Lógicos Bit a Bit (Obligatorios en NumPy)
Para combinar múltiples condiciones booleanas sobre arreglos, **no** se usan las palabras clave de Python (`and`, `or`, `not`), sino operadores bit a bit con paréntesis obligatorios alrededor de cada condición:
* **`&`** : AND lógico (ambas condiciones deben cumplirse).
* **`|`** : OR lógico (al menos una condición debe cumplirse).
* **`~`** : NOT lógico (niega o invierte la máscara booleana).

### Otras Herramientas de Selección Avanzada
* **`np.where(condición, valor_si_true, valor_si_false)`:** Evalúa una condición y construye un nuevo arreglo manteniendo el `shape` original (ideal para recodificar datos).
* **Fancy Indexing:** Pasar un arreglo de enteros como índice (`arr[[0, 3, 1]]`) para extraer filas o columnas específicas en cualquier orden. (Nota: Tanto las máscaras booleanas como el *Fancy Indexing* siempre devuelven una **copia**, nunca una vista).

### Sintaxis Mínima

In [17]:
# Matriz 4x5: Niveles de concentración de contaminantes (ppm) en 4 zonas y 5 mediciones
concentraciones = np.array([
    [12.4, 18.9, 45.2, 51.0, 22.1],
    [8.5,  9.2,  11.0, 14.3, 10.1],
    [35.0, 42.8, 60.5, 58.2, 49.0],
    [19.0, 21.5, 29.8, 33.4, 27.6]
])

print("Matriz original\n", concentraciones)

# 1. Máscara booleana simple: valores críticos mayores a 40 ppm
mascara_critica = concentraciones > 40.0
print("\nMáscara booleana (shape", mascara_critica.shape, "):\n", mascara_critica)

# Al aplicar la máscara, NumPy extrae los valores que cumplen en un vector 1D
print("\nValores críticos extraídos:", concentraciones[mascara_critica])

# 2. Combinación de condiciones: valores moderados entre 20 y 40 ppm (inclusive)
moderados = concentraciones[(concentraciones >= 20.0) & (concentraciones <= 40.0)]
print("Valores moderados [20, 40]:", moderados)

# 3. Modificación condicional in-place usando máscara: acotar picos mayores a 50 al límite de 50
datos_limpios = concentraciones.copy()
datos_limpios[datos_limpios > 50.0] = 50.0
print("\nMatriz tras acotar valores > 50:\n", datos_limpios)

# 4. np.where: Clasificar cada celda como 'ALERTA' (>= 35) o 'NORMAL' (< 35) manteniendo shape (4, 5)
etiquetas = np.where(concentraciones >= 35.0, "ALERTA", "NORMAL")
print("\nMatriz de diagnóstico (np.where):\n", etiquetas)

# 5. Fancy Indexing: Extraer únicamente las zonas 2, 0 y 3 en ese orden específico
zonas_prioritarias = concentraciones[[2, 0, 3], :]
print("\nZonas reordenadas (Fancy Indexing):\n", zonas_prioritarias)

Matriz original
 [[12.4 18.9 45.2 51.  22.1]
 [ 8.5  9.2 11.  14.3 10.1]
 [35.  42.8 60.5 58.2 49. ]
 [19.  21.5 29.8 33.4 27.6]]

Máscara booleana (shape (4, 5) ):
 [[False False  True  True False]
 [False False False False False]
 [False  True  True  True  True]
 [False False False False False]]

Valores críticos extraídos: [45.2 51.  42.8 60.5 58.2 49. ]
Valores moderados [20, 40]: [22.1 35.  21.5 29.8 33.4 27.6]

Matriz tras acotar valores > 50:
 [[12.4 18.9 45.2 50.  22.1]
 [ 8.5  9.2 11.  14.3 10.1]
 [35.  42.8 50.  50.  49. ]
 [19.  21.5 29.8 33.4 27.6]]

Matriz de diagnóstico (np.where):
 [['NORMAL' 'NORMAL' 'ALERTA' 'ALERTA' 'NORMAL']
 ['NORMAL' 'NORMAL' 'NORMAL' 'NORMAL' 'NORMAL']
 ['ALERTA' 'ALERTA' 'ALERTA' 'ALERTA' 'ALERTA']
 ['NORMAL' 'NORMAL' 'NORMAL' 'NORMAL' 'NORMAL']]

Zonas reordenadas (Fancy Indexing):
 [[35.  42.8 60.5 58.2 49. ]
 [12.4 18.9 45.2 51.  22.1]
 [19.  21.5 29.8 33.4 27.6]]


:::{warning} Trampa Común: Usar `and` / `or` u olvidar los paréntesis
Si escribes `arr[arr > 10 and arr < 20]`, Python lanzará `ValueError: The truth value of an array with more than one element is ambiguous`.
Asimismo, si escribes `arr[arr > 10 & arr < 20]` **sin paréntesis**, Python evaluará primero `10 & arr` por precedencia de operadores y fallará. Usa siempre paréntesis: `arr[(arr > 10) & (arr < 20)]`.
:::

:::{note} Restricción de Forma al Filtrar con Máscaras Booleanas
Cuando aplicas una máscara 2D sobre una matriz 2D (`matriz[mascara_2d]`), el resultado **siempre se aplana a un vector 1D**, porque los elementos que cumplen la condición rara vez forman un rectángulo perfecto. Si necesitas conservar la estructura 2D, utiliza `np.where()`.
:::

### 🧩 Micro-reto 2.3
Dada la matriz `temperaturas_horno` de $4 \times 5$ generada en la celda siguiente:
1. Extrae en un vector 1D todas las temperaturas que sean **menores a 180 °C o mayores a 230 °C** (fuera de rango óptimo).
2. Utiliza `np.where()` para crear una matriz del mismo tamaño (`4x5`) donde los valores fuera de rango (`< 180` o `> 230`) se reemplacen por `0` y los valores dentro del rango óptimo conserven su temperatura original.

In [18]:
temperaturas_horno = np.array([
    [175, 185, 200, 215, 240],
    [190, 195, 205, 220, 225],
    [160, 178, 210, 235, 250],
    [182, 198, 212, 228, 230]
])

# Escribe tu solución aquí:

## ✅ Resumen del Módulo 2
* Indexa siempre matrices 2D con una sola pareja de corchetes: `matriz[filas, columnas]`.
* El *slicing* devuelve una **vista (*View*)** compartida en memoria; usa `.copy()` siempre que necesites modificar subconjuntos sin alterar los datos originales.
* Combina filtros booleanos usando **`&`** (AND), **`|`** (OR) y **`~`** (NOT) encerrando cada condición entre paréntesis.
* Emplea `np.where(condición, A, B)` cuando necesites transformar valores condicionalmente preservando las dimensiones originales de la matriz.